# Test Dataset

This page describes how the test datasets were generated and how they are used in the tests.

## Scaling workflow - MTZ files

MTZ test datasets are created with ``gemmi`` and a random-number generator.

We have multiple test MTZ files since multiple files are expected in usual cases.

These files do not have any physical meaning and they are meant to be useful for testing the workflow.

Here is the code cell to create the test files.

In [ ]:
import gemmi
import pandas as pd
import numpy as np
from ess.nmx.mtz_io import (
    DEFAULT_INTENSITY_COLUMN_NAME,
    DEFAULT_WAVELENGTH_COLUMN_NAME,
    DEFAULT_STD_DEV_COLUMN_NAME,
    DEFAULT_SPACE_GROUP_DESC,
)

# Negative intensities will happen due to corrections
# and high intensities are also expected in some cases
INTENSITY_RANGE = (-20.0, 200.0)
HKL_RANGE = (-100, 100)
MANDATORY_FIELDS = (
    "H",
    "K",
    "L",
    DEFAULT_WAVELENGTH_COLUMN_NAME,  # LAMBDA
    DEFAULT_INTENSITY_COLUMN_NAME,  # I
    DEFAULT_STD_DEV_COLUMN_NAME,  # SIGI
)
global_rng = np.random.default_rng(0)
HKL_CANDIDATES = tuple(
    zip(*[global_rng.integers(*HKL_RANGE, size=100) for _ in range(3)], strict=False)
)

def create_mtz_data_frame(random_seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(random_seed)
    intensities = np.sort(rng.normal(50, 20, size=10_000))[::-1] + (
        rng.uniform(*INTENSITY_RANGE, size=10_000)
        * rng.choice([0] * 99 + [1], size=10_000)
    )
    std_devs = np.multiply(intensities, rng.uniform(0.1, 0.15, size=10_000))
    wavelengths = np.sort(rng.uniform(2.8, 3.2, size=10_000))[::-1]

    df = pd.DataFrame(
        {
            DEFAULT_INTENSITY_COLUMN_NAME: intensities,
            DEFAULT_STD_DEV_COLUMN_NAME: std_devs,
            DEFAULT_WAVELENGTH_COLUMN_NAME: wavelengths,
        }
    )

    df[["H", "K", "L"]] = pd.Series(
        rng.choice(HKL_CANDIDATES, size=10_000).tolist()
    ).to_list()

    return df


def dataframe_to_mtz(df: pd.DataFrame) -> gemmi.Mtz:
    """Create a random MTZ file with a single dataset.

    Columns:
    - H, K, L: Miller indices
    - LAMBDA: Wavelength
    - I: Intensity
    - SIGI: Standard deviation of intensity

    """
    assert set(df.columns) == set(MANDATORY_FIELDS)

    mtz = gemmi.Mtz()
    mtz.add_dataset("HKL")
    column_type_map = {  # Column types: https://www.ccp4.ac.uk/html/mtzformat.html#coltypes
        "H": "H",
        "K": "H",
        "L": "H",
        DEFAULT_WAVELENGTH_COLUMN_NAME: "R",
        DEFAULT_INTENSITY_COLUMN_NAME: "J",
        DEFAULT_STD_DEV_COLUMN_NAME: "Q",
    }

    for col_name in df.columns:
        mtz.add_column(col_name, type=column_type_map[col_name], expand_data=True)

    mtz.spacegroup = gemmi.SpaceGroup(DEFAULT_SPACE_GROUP_DESC)
    mtz.set_data(df.values)
    return mtz


for seed in range(1, 6):
    sample_df = create_mtz_data_frame(seed)
    sample_mtz = dataframe_to_mtz(sample_df)
    # sample_mtz.write_to_file(f"sample_{seed}.mtz")  # Uncomment to save the MTZ file

sample_df.head()

Once the files were created, they were compressed into one file
and uploaded in the server where pooch can access to.

Here is the script for compressing the files.

```bash
tar -czvf mtz_random_samples.tar.gz sample_*.mtz
```

## Regression Test Dataset

There are frozen reduced output files for output regression tests. They can be downloaded via pooch using ess.nmx.data module. Frozen reduced files have prefix `essnmx-reduced`.

The change of output also depends on the package dependency of `essnmx`, for example `essreduce` or `tof`.
The regression test guarantees that the output with the lower-bound resolutions or any dev-environment locked environment can reproduce the same output as the frozen ones.

However, it does not guanratee that the `essnmx` will produce the same output with the latest dependencies.
We try to detect if any upgrades of dependencies breaks the regression test.
In that case we increase the minimum version of that dependency and update the frozen reduced file using the upgraded dependency.

i.e. whenever the regression tests fails in the latest dependency nightly, maintainers should check
- Why the result is changed.
- How much the result is changed.
- If the change is acceptable.
    - If True, the frozen reduced file and the minimum version of the dependency need to be updated.

### Freezing reduced data.

There is a script to generate a frozen data from the small mcstas nexus (sampled events data from mcstas results was injected into a template nexus file.)
A new reduced results can be frozen:

```bash
pixi run --frozen python tools/freeze-reduced.py --freeze-version 2*.*.*
```

Note that the new version of essnmx(==freeze-version) should be released including the pixi(python) lock file.
It will reduce a lot of debugging efforts if we know all the exact versions of the dependencies that the reduced file was generated with.

### Change log

| Version | Description |
| - | - |
| 26.4.1 | First frozen version. Most of previous versions of `essnmx` should also produce the same output with the default settings. (except for a few versions between >26.0.0,<26.4.1) |
| 26.6.1 | Lower bound `tof>=26.6.0` version introduced changes in the output. |

### Version Diffs

We keep the frozen reduced version for 2 years (no specific reason) and keep track of the diffs 

In [ ]:
from ess.nmx.data import get_path, _registry
from ess.nmx._nxlauetof_io import load_essnmx_nxlauetof
import plopp as pp
import scipp as sc

frozen_reduced_files = tuple(file_name for file_name in _registry._files if file_name.startswith("essnmx-reduced-"))
results = {frozen: load_essnmx_nxlauetof(get_path(frozen)) for frozen in frozen_reduced_files}

tiled_panels = pp.tiled(ncols=3, nrows=1)
tiled_panels.fig.suptitle("Frozen reduced results comparison")
all_panels_hists = {}
for detector_panel_id in range(3):
    panel_name = f'detector_panel_{detector_panel_id}'
    hists = {frozen: dg['instrument']['detectors'][panel_name]['data'].sum(('x_pixel_offset', 'y_pixel_offset')) for frozen, dg in results.items()}
    for frozen, hist in hists.items():
        all_panels_hists.setdefault(frozen, sc.zeros_like(hist))
        all_panels_hists[frozen] += hist
    tiled_panels[0, detector_panel_id] = pp.plot(hists, grid=True, title=f"{panel_name}")

display(tiled_panels)
pp.plot(all_panels_hists, grid=True, title="Frozen reduced results (all panels combined)")